# Phase 4 — Demand Forecasting & Anomaly Detection

Builds time-series demand forecasts and flags anomalous transactions on top of Phase 3 outputs.

**Sections**
1. Setup & data loading
2. Step 4.1 — Demand Forecasting (SARIMA + Prophet)
3. Step 4.2 — Anomaly Detection (Isolation Forest)
4. Phase 4 Summary

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from models.data_pipeline import load_raw_data, extract_signal_before_cleaning, clean_data
from models.forecast_model import (
    build_weekly_revenue,
    run_adf_test,
    fit_sarima,
    forecast_sarima,
    fit_prophet_model,
    plot_dual_forecast,
    save_demand_forecasts,
    build_anomaly_features,
    fit_isolation_forest,
    score_anomalies,
    save_anomaly_flags,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
_, df_all = clean_data(df_raw)
print(f'df_all shape: {df_all.shape}')
print(df_all.dtypes)

## Step 4.1 — Demand Forecasting

**Approach:** Weekly revenue time series per product category. Two complementary models:
- **SARIMA(1,1,1)(1,1,0,52)** — classical statistical model capturing weekly seasonality
- **Prophet** — decomposable trend model with yearly + weekly seasonality

**Business question:** How will revenue trend over the next 12 weeks per category, and where do the two models agree or diverge?

In [ ]:
weekly_df, top5 = build_weekly_revenue(df_all)
print('Top 5 categories by total revenue:')
print(top5)
print(f'
weekly_df shape: {weekly_df.shape}')
print(weekly_df.head())

### Stationarity Check — ADF Test

The Augmented Dickey-Fuller test checks whether each category's time series has a unit root (non-stationary). If p > 0.05 we fail to reject the null of a unit root and apply differencing before fitting SARIMA.

In [ ]:
adf_results = {}
for cat in top5:
    series = weekly_df[weekly_df['category'] == cat].set_index('week')['revenue']
    p = run_adf_test(series, cat)
    adf_results[cat] = p

print('
ADF p-values:')
for cat, p in adf_results.items():
    status = 'stationary' if p <= 0.05 else 'non-stationary → will difference'
    print(f'  {cat}: p={p:.4f}  ({status})')

### SARIMA Models

SARIMAX order=(1,1,1) seasonal_order=(1,1,0,52) — one autoregressive, one differencing, one moving-average term, with seasonal period of 52 weeks.

In [ ]:
sarima_forecasts = {}
sarima_history = {}

for cat in top5:
    series = weekly_df[weekly_df['category'] == cat].set_index('week')['revenue']
    results = fit_sarima(series, cat)
    fc = forecast_sarima(results, steps=12)
    sarima_forecasts[cat] = fc
    sarima_history[cat] = series
    print(f'{cat}: forecast range £{fc["sarima_forecast"].min():.0f} – £{fc["sarima_forecast"].max():.0f}')

### Prophet Models

Prophet handles missing data and outliers well. `yearly_seasonality=True` and `weekly_seasonality=True` capture both seasonal patterns present in retail data.

In [ ]:
prophet_forecasts = {}

for cat in top5:
    series = weekly_df[weekly_df['category'] == cat].set_index('week')['revenue']
    fc = fit_prophet_model(series, cat)
    prophet_forecasts[cat] = fc
    print(f'{cat}: Prophet yhat range £{fc["yhat"].min():.0f} – £{fc["yhat"].max():.0f}')

### Dual Forecast Comparison Plots

Each chart shows historical weekly revenue (black), SARIMA forecast with 95% CI (blue shading), and Prophet forecast (red dashed). Where both models agree we have higher confidence in the projection.

In [ ]:
for cat in top5:
    plot_dual_forecast(
        category=cat,
        history=sarima_history[cat],
        sarima_fc=sarima_forecasts[cat],
        prophet_fc=prophet_forecasts[cat],
    )

In [ ]:
records = []
for cat in top5:
    s_fc = sarima_forecasts[cat]
    p_fc = prophet_forecasts[cat].reset_index(drop=True)
    for i in range(len(s_fc)):
        records.append({
            'category':         cat,
            'date':             str(p_fc['ds'].iloc[i].date()),
            'sarima_forecast':  round(s_fc['sarima_forecast'].iloc[i], 2),
            'prophet_forecast': round(p_fc['yhat'].iloc[i], 2),
            'lower_ci':         round(s_fc['lower_ci'].iloc[i], 2),
            'upper_ci':         round(s_fc['upper_ci'].iloc[i], 2),
        })

save_demand_forecasts(records)
print(pd.DataFrame(records).head(10))

## Step 4.2 — Anomaly Detection

**Model:** Isolation Forest with contamination=0.01 (expects ~1% of transactions to be anomalous).

**Features:**
| Feature | Description |
|---|---|
| order_value | quantity × unit_price |
| order_value_zscore | standardised order value per customer |
| quantity | items ordered |
| hour_of_day | transaction hour (0–23) |
| day_of_week | 0=Mon … 6=Sun |
| is_new_country | 1 if country ≠ customer primary country |

**Business question:** Which transactions deviate significantly from normal purchasing behaviour and may indicate fraud, data errors, or bulk orders?

In [ ]:
X, df_flagged = build_anomaly_features(df_all)
print(f'feature matrix shape: {X.shape}')
feat_cols = ["order_value","order_value_zscore","quantity","hour_of_day","day_of_week","is_new_country"]
print(pd.DataFrame(X, columns=feat_cols).describe().round(3))